In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from torchvision.transforms import v2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")


# Define transformations for the training and validation sets


transform_train = v2.Compose([
    v2.ToImage(),
    v2.RandomResizedCrop((224, 224)),
    v2.RandomHorizontalFlip(),
    v2.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    v2.ToDtype(torch.float32,scale = True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_val = v2.Compose([
    v2.ToImage(),
    v2.Resize((256, 256)),
    v2.CenterCrop((224, 224)),
    v2.ToDtype(torch.float32,scale = True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
# load oxford iiit pets dataset
train_dataset = datasets.OxfordIIITPet(root = './data',split = 'trainval',transform = transform_train, download = True)
val_dataset = datasets.OxfordIIITPet(root = './data', split = 'test', transform = transform_val, download = True)

train_loader = DataLoader(train_dataset, batch_size = 128, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size = 128, shuffle = False)


0.0%


KeyboardInterrupt: 

In [ ]:
model = models.resnet50(weights = models.ResNet50_Weights.DEFAULT)
for params in model.parameters():
    params.requires_grad = False

input_features = model.fc.in_features
model.fc = nn.Linear(input_features,37)
model = model.to(device)
model

In [ ]:
epochs = 10 
criterion  = nn.CrossEntropyLoss()
optimizer = optim.Adam([{"params": model.fc.parameters(), "lr": 0.01},
                        {"params" : model.layer4.parameters(), "lr":0.001},
                        {"params" : model.layer3.parameters(), "lr": 0.0001},
                        {"params": model.layer2.parameters(), "lr": 0.0001}])
# here we have added different lr for different layers of the model. The final layer has the highest lr because it is randomly initialized and needs to learn the most, while the initial layers have the lowest lr because they are pre-trained and only need to be fine-tuned slightly.
step_scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=4,gamma= 0.1)
plateau_scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode = 'min', factor = 0.5, patience = 2)


In [ ]:
training_loss = []
for epoch in range(epochs):
    model.train()
    trn_loss = 0
    i = 0
    if epoch == 2:
        for param in model.layer4.parameters():
            params.required_grad = True
        print("Unfroze layer 4")

    if epoch == 5:
        for params in model.layer3.parameters():
            params.requires_grad = True
        print("Unfroze layer 3")
    
    if epoch == 8:
        for params in model.layer2.parameters():
            params.requires_grad = True
        print("Unfroze layer 2")
    for batch_X, batch_y in train_loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)
        y_pred = model(batch_X)
        loss = criterion(y_pred, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        trn_loss += loss.item()
        i += 1
        print(f"Batch {i}, Loss: {loss.item():.4f}")
        
    average_epoch_loss = trn_loss / len(train_loader)
    training_loss.append(average_epoch_loss)
    print(f"Epoch {epoch+1}/{epochs}, Training Loss: {average_epoch_loss:.10f}")
    step_scheduler.step()
    plateau_scheduler.step(average_epoch_loss)

In [ ]:
correct = 0; 
test_len = len(val_dataset)
model.eval()
with torch.no_grad():
    for batch_X,batch_y in val_loader:
        batch_correct_preds = 0
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)
        y_pred = model(batch_X)
        pred_labels = torch.argmax(y_pred,dim =1)
        batch_correct_preds += (pred_labels== batch_y).sum().item()
        correct += batch_correct_preds


print(correct)
accuracy = correct/test_len
print(f"Validation Accuracy: {accuracy*100:.10f}%")